# reduce-op-mean-divide — worked example 2: Geometric mean via log → all_reduce → divide → exp

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reduce-op-mean-divide`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Many cross-rank statistics reduce to the sum-then-divide pattern after a transform. The geometric mean is `exp(mean(log x))`: take the log of each rank's value, `all_reduce(SUM)`, divide by `world_size` to get the mean of logs, then exponentiate. This works because the log turns products into sums, which `ReduceOp.SUM` can handle.

## Worked solution

Each rank holds a positive scalar. `geometric_mean` builds `tensor = [log(local_value)]`, calls `all_reduce(SUM)` so the tensor holds the sum of logs, divides in place by `world_size` to get the mean of the logs, then returns `exp(tensor.item())`. We validate the input is positive first, since log is undefined otherwise. The transform-sum-divide-invert shape mirrors the harmonic-mean idiom but with log/exp instead of reciprocal. We print the geometric mean of [1,4,16], which should be `(1*4*16)**(1/3) = 4.0`.

In [ ]:
import math


class FakeDist:
    def __init__(self, log_contribs):
        self.total = sum(log_contribs)
    def all_reduce(self, tensor, op='sum'):
        tensor.copy_(t.tensor([self.total]))


def geometric_mean(world_size, local_value, dist_module):
    if local_value <= 0:
        raise ValueError(f'geometric mean undefined for non-positive: {local_value}')
    tensor = t.tensor([math.log(local_value)])
    dist_module.all_reduce(tensor, op='sum')
    tensor /= world_size
    return math.exp(tensor.item())


values = [1.0, 4.0, 16.0]
fd = FakeDist([math.log(v) for v in values])
print('geometric mean:', round(geometric_mean(len(values), 1.0, fd), 6))